<a href="https://colab.research.google.com/github/abmila/Scraper-de-coopel-local-para-codex/blob/main/MACRO_FINANCIERO_5D_US_MX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 ESTUDIO MACRO-FINANCIERO 5 DÍAS — EUA + MÉXICO
### Sistema de análisis cuantitativo con Monte Carlo ≥ 1,000,000 simulaciones
**Ejecutar: Runtime → Run all (sin inputs manuales requeridos)**

---
- **Autor:** Senior Quant Research Engineer + Macro Portfolio Strategist
- **Universo:** SPY, QQQ, GLD, SLV, Sectores S&P500 SPDR, USD/MXN, DXY, WTI, VIX
- **Macro:** FRED (fuente oficial)
- **Modelos:** kNN, Logistic Regression, Monte Carlo VAR(1) con batching
- **Output:** Excel multi-hoja + conclusiones en español

In [1]:
# ============================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

for pkg in ['yfinance', 'pandas_datareader', 'openpyxl', 'scikit-learn', 'statsmodels', 'numpy', 'pandas']:
    install(pkg)

print('✅ Librerías instaladas.')

✅ Librerías instaladas.


In [2]:
# ============================================================
# CELDA 2: PARÁMETROS GLOBALES
# ============================================================
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

TODAY = pd.Timestamp.today().normalize()
END = TODAY
FOCUS_START = TODAY - pd.DateOffset(years=5)
DOWNLOAD_START = TODAY - pd.DateOffset(years=12)

HORIZON_BDAYS = 5       # horizonte de pronóstico
N_SIM = 1_000_000       # simulaciones Monte Carlo totales
BATCH_SIZE = 50_000     # tamaño de lote
K_NEIGHBORS = 200       # vecinos kNN
SEED = 42

EXCEL_FILE = 'ESTUDIO_MACRO_5D_US_MX_MONTECARLO.xlsx'

print(f'📅 Fecha hoy        : {TODAY.date()}')
print(f'📅 Inicio análisis  : {FOCUS_START.date()}')
print(f'📅 Inicio descarga  : {DOWNLOAD_START.date()}')
print(f'🎲 Simulaciones MC  : {N_SIM:,}')
print(f'🔢 Lote MC          : {BATCH_SIZE:,}')

📅 Fecha hoy        : 2026-03-11
📅 Inicio análisis  : 2021-03-11
📅 Inicio descarga  : 2014-03-11
🎲 Simulaciones MC  : 1,000,000
🔢 Lote MC          : 50,000


In [3]:
# ============================================================
# CELDA 3: UNIVERSO DE ACTIVOS Y NOMBRES COMPLETOS
# ============================================================

# Diccionario: clave_interna -> (nombre_completo, [tickers_en_orden_de_preferencia])
UNIVERSE = {
    'SPY':   ('S&P 500 (ETF SPY)',                        ['SPY']),
    'QQQ':   ('Nasdaq 100 (ETF QQQ)',                     ['QQQ', '^NDX']),
    'GLD':   ('Oro (ETF GLD)',                            ['GLD', 'GC=F']),
    'SLV':   ('Plata (ETF SLV)',                          ['SLV', 'SI=F']),
    'XLC':   ('Sector Comunicaciones (XLC)',              ['XLC']),
    'XLY':   ('Sector Consumo Discrecional (XLY)',        ['XLY']),
    'XLP':   ('Sector Consumo Básico (XLP)',              ['XLP']),
    'XLE':   ('Sector Energía (XLE)',                     ['XLE']),
    'XLF':   ('Sector Financiero (XLF)',                  ['XLF']),
    'XLV':   ('Sector Salud (XLV)',                       ['XLV']),
    'XLI':   ('Sector Industrial (XLI)',                  ['XLI']),
    'XLB':   ('Sector Materiales (XLB)',                  ['XLB']),
    'XLRE':  ('Sector Bienes Raíces (XLRE)',              ['XLRE']),
    'XLK':   ('Sector Tecnología (XLK)',                  ['XLK']),
    'XLU':   ('Sector Servicios Públicos (XLU)',          ['XLU']),
    'USDMXN':('Tipo de Cambio USD/MXN',                  ['MXN=X']),
    'DXY':   ('Índice Dólar (DXY)',                       ['DX-Y.NYB', '^DXY']),
    'WTI':   ('Petróleo WTI (Proxy USO)',                 ['CL=F', 'USO']),
    'VIX':   ('Índice de Volatilidad VIX',               ['^VIX']),
}

# Nombres completos para outputs
FULL_NAME = {k: v[0] for k, v in UNIVERSE.items()}

# Activos de inversión (excluye FX y VIX del target)
ASSET_KEYS = ['SPY','QQQ','GLD','SLV','XLC','XLY','XLP','XLE',
              'XLF','XLV','XLI','XLB','XLRE','XLK','XLU']

# Endógenos para VAR (variables de estado del mercado)
ENDOG_KEYS = ['USDMXN','DXY','WTI','SPY']

print(f'📋 Universo total   : {len(UNIVERSE)} activos/series')
print(f'📈 Activos target   : {len(ASSET_KEYS)}')

📋 Universo total   : 19 activos/series
📈 Activos target   : 15


In [4]:
# ============================================================
# CELDA 4: DESCARGA YAHOO FINANCE — ROBUSTA CON FALLBACKS
# ============================================================
import yfinance as yf
import time

def download_ticker(ticker, start, end, min_rows=200, retries=3):
    """Descarga un ticker con reintentos. Retorna Series de precios Adj Close o Close."""
    for attempt in range(retries):
        try:
            df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if df is None or df.empty:
                time.sleep(1)
                continue
            # Extraer columna de precio
            if 'Close' in df.columns:
                s = df['Close'].squeeze()
            elif 'Adj Close' in df.columns:
                s = df['Adj Close'].squeeze()
            else:
                s = df.iloc[:, 0].squeeze()
            s = s.dropna()
            if len(s) >= min_rows:
                return s
        except Exception as e:
            print(f'    ⚠️  {ticker} intento {attempt+1} error: {e}')
            time.sleep(2)
    return None

def download_by_key(key, tickers, start, end, min_rows=200):
    """Prueba tickers en orden hasta encontrar uno válido."""
    for ticker in tickers:
        print(f'  → Intentando {key}: {ticker}...', end=' ')
        s = download_ticker(ticker, start, end, min_rows)
        if s is not None and len(s) >= min_rows:
            print(f'✅ ({len(s)} filas)')
            return s, ticker
        print('❌')
    return None, None

# Descarga de todos los activos
prices_raw = {}       # key -> Series de precios
chosen_tickers = {}   # key -> ticker elegido
missing_yahoo = []    # keys que fallaron

print('🔽 Descargando precios Yahoo Finance...')
for key, (fullname, tickers) in UNIVERSE.items():
    s, chosen = download_by_key(key, tickers, DOWNLOAD_START, END)
    if s is not None:
        prices_raw[key] = s
        chosen_tickers[key] = chosen
    else:
        missing_yahoo.append(key)
        print(f'  ⛔ {key} ({fullname}): NO SE PUDO DESCARGAR')

print(f'\n✅ Descargados: {len(prices_raw)}/{len(UNIVERSE)}')
print(f'❌ Faltantes: {missing_yahoo}')

# Construir DataFrame de precios alineado a días hábiles
prices_df = pd.DataFrame(prices_raw)
prices_df.index = pd.to_datetime(prices_df.index)
prices_df = prices_df.sort_index()

# Índice maestro: días hábiles en rango de descarga
bday_index = pd.bdate_range(start=DOWNLOAD_START, end=END)
prices_df = prices_df.reindex(bday_index).ffill().dropna(how='all')

print(f'\n📊 prices_df shape: {prices_df.shape}')
print(f'   Rango: {prices_df.index[0].date()} → {prices_df.index[-1].date()}')

🔽 Descargando precios Yahoo Finance...
  → Intentando SPY: SPY... ✅ (3018 filas)
  → Intentando QQQ: QQQ... ✅ (3018 filas)
  → Intentando GLD: GLD... ✅ (3018 filas)
  → Intentando SLV: SLV... ✅ (3018 filas)
  → Intentando XLC: XLC... ✅ (1941 filas)
  → Intentando XLY: XLY... ✅ (3018 filas)
  → Intentando XLP: XLP... ✅ (3018 filas)
  → Intentando XLE: XLE... ✅ (3018 filas)
  → Intentando XLF: XLF... ✅ (3018 filas)
  → Intentando XLV: XLV... ✅ (3018 filas)
  → Intentando XLI: XLI... ✅ (3018 filas)
  → Intentando XLB: XLB... ✅ (3018 filas)
  → Intentando XLRE: XLRE... ✅ (2619 filas)
  → Intentando XLK: XLK... ✅ (3018 filas)
  → Intentando XLU: XLU... ✅ (3018 filas)
  → Intentando USDMXN: MXN=X... ✅ (3125 filas)
  → Intentando DXY: DX-Y.NYB... ✅ (3019 filas)
  → Intentando WTI: CL=F... ✅ (3018 filas)
  → Intentando VIX: ^VIX... ✅ (3018 filas)

✅ Descargados: 19/19
❌ Faltantes: []

📊 prices_df shape: (3132, 19)
   Rango: 2014-03-11 → 2026-03-11


In [5]:
# ============================================================
# CELDA 5: DESCARGA FRED — ROBUSTA CON FALLBACKS
# ============================================================
from pandas_datareader import data as pdr

FRED_CANDIDATES = {
    # Tasas EUA
    'EFFR':         ['EFFR', 'DFF'],
    'DGS10':        ['DGS10'],
    'DGS2':         ['DGS2'],
    'DGS3MO':       ['TB3MS', 'DGS3MO'],
    # Inflación EUA
    'CPI':          ['CPIAUCSL'],
    'CORECPI':      ['CPILFESL'],
    'PCE':          ['PCEPI'],
    'COREPCE':      ['PCEPILFE'],
    'BREAKEVEN5Y':  ['T5YIE'],
    'BREAKEVEN10Y': ['T10YIE'],
    # Actividad / Empleo EUA
    'UNRATE':       ['UNRATE'],
    'CLAIMS':       ['ICSA'],
    'RETAIL':       ['RSXFS', 'RSALES'],
    'REALPCE':      ['PCEC96'],
    'INDPRO':       ['INDPRO'],
    'SENTIMENT':    ['UMCSENT'],
    'ISM_PMI':      ['NAPM', 'MANEMP'],
    # Riesgo / Condiciones financieras EUA
    'NFCI':         ['NFCI'],
    'HY_SPREAD':    ['BAMLH0A0HYM2'],
    'BAA':          ['BAA'],
    # Liquidez
    'M2':           ['M2SL'],
    # DXY proxy FRED
    'TWD':          ['DTWEXBGS', 'DTWEXM'],
    # México
    'MX_CPI':       ['CPALTT01MXM659N'],
    'MX_3M':        ['IR3TIB01MXM156N'],
    'MX_10Y':       ['IRLTLT01MXM156N'],
}

# Frecuencias conocidas para lags anti-lookahead
# 'M'=mensual, 'W'=semanal, 'D'=diaria
FRED_FREQ = {
    'EFFR':'D','DGS10':'D','DGS2':'D','DGS3MO':'D',
    'CPI':'M','CORECPI':'M','PCE':'M','COREPCE':'M',
    'BREAKEVEN5Y':'D','BREAKEVEN10Y':'D',
    'UNRATE':'M','CLAIMS':'W','RETAIL':'M','REALPCE':'M',
    'INDPRO':'M','SENTIMENT':'M','ISM_PMI':'M',
    'NFCI':'W','HY_SPREAD':'D','BAA':'D','M2':'M',
    'TWD':'D',
    'MX_CPI':'M','MX_3M':'M','MX_10Y':'M',
}

def fetch_fred(series_id, start, end):
    try:
        s = pdr.DataReader(series_id, 'fred', start=start, end=end)
        s = s.squeeze().dropna()
        if len(s) > 10:
            return s
    except Exception as e:
        pass
    return None

macro_raw = {}
chosen_fred = {}
missing_fred = []

print('🔽 Descargando indicadores FRED...')
for key, candidates in FRED_CANDIDATES.items():
    found = False
    for sid in candidates:
        print(f'  → {key}: {sid}...', end=' ')
        s = fetch_fred(sid, DOWNLOAD_START, END)
        if s is not None:
            macro_raw[key] = s
            chosen_fred[key] = sid
            print(f'✅ ({len(s)} obs)')
            found = True
            break
        print('❌')
    if not found:
        missing_fred.append(key)
        print(f'  ⛔ {key}: NO DISPONIBLE EN FRED')

print(f'\n✅ FRED descargados: {len(macro_raw)}/{len(FRED_CANDIDATES)}')
print(f'❌ FRED faltantes: {missing_fred}')

🔽 Descargando indicadores FRED...
  → EFFR: EFFR... ✅ (3012 obs)
  → DGS10: DGS10... ✅ (2999 obs)
  → DGS2: DGS2... ✅ (2999 obs)
  → DGS3MO: TB3MS... ✅ (143 obs)
  → CPI: CPIAUCSL... ✅ (141 obs)
  → CORECPI: CPILFESL... ✅ (141 obs)
  → PCE: PCEPI... ✅ (141 obs)
  → COREPCE: PCEPILFE... ✅ (141 obs)
  → BREAKEVEN5Y: T5YIE... ✅ (3000 obs)
  → BREAKEVEN10Y: T10YIE... ✅ (3000 obs)
  → UNRATE: UNRATE... ✅ (142 obs)
  → CLAIMS: ICSA... ✅ (625 obs)
  → RETAIL: RSXFS... ✅ (142 obs)
  → REALPCE: PCEC96... ✅ (141 obs)
  → INDPRO: INDPRO... ✅ (142 obs)
  → SENTIMENT: UMCSENT... ✅ (142 obs)
  → ISM_PMI: NAPM... ❌
  → ISM_PMI: MANEMP... ✅ (143 obs)
  → NFCI: NFCI... ✅ (625 obs)
  → HY_SPREAD: BAMLH0A0HYM2... ✅ (3135 obs)
  → BAA: BAA... ✅ (143 obs)
  → M2: M2SL... ✅ (142 obs)
  → TWD: DTWEXBGS... ✅ (2988 obs)
  → MX_CPI: CPALTT01MXM659N... ✅ (124 obs)
  → MX_3M: IR3TIB01MXM156N... ✅ (142 obs)
  → MX_10Y: IRLTLT01MXM156N... ✅ (108 obs)

✅ FRED descargados: 25/25
❌ FRED faltantes: []


In [6]:
# ============================================================
# CELDA 6: ALINEACIÓN + LAGS ANTI-LOOKAHEAD
# ============================================================

# Usamos el índice de precios como base (días hábiles)
bday_idx = prices_df.index  # ya está en bdays

LAG_BDAYS = {'M': 21, 'W': 5, 'D': 0}  # lag conservador por frecuencia

macro_aligned = {}

for key, series in macro_raw.items():
    freq = FRED_FREQ.get(key, 'M')  # default mensual (más conservador)
    lag = LAG_BDAYS[freq]

    # Convertir índice a DatetimeIndex
    s = series.copy()
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()

    # Reindex a días hábiles
    s_bday = s.reindex(bday_idx).ffill()

    # Aplicar lag ANTES del ffill final (anti-lookahead)
    # Lag = desplazar hacia adelante: el dato de hoy no está disponible hasta hoy+lag
    if lag > 0:
        s_bday = s_bday.shift(lag)

    macro_aligned[key] = s_bday

# DataFrame macro alineado
macro_df = pd.DataFrame(macro_aligned, index=bday_idx)

# Construir YoY para series de nivel mensual
def yoy_change(key, df, lag_offset=0):
    """Cambio YoY: (hoy - hace 252 bdays) / hace 252 bdays"""
    if key in df.columns:
        s = df[key]
        denom = s.shift(252).abs()
        return (s - s.shift(252)) / denom.where(denom > 1e-8) * 100
    return pd.Series(np.nan, index=df.index)

macro_df['CPI_YOY']     = yoy_change('CPI', macro_df)
macro_df['CORECPI_YOY'] = yoy_change('CORECPI', macro_df)
macro_df['PCE_YOY']     = yoy_change('PCE', macro_df)
macro_df['M2_YOY']      = yoy_change('M2', macro_df)
macro_df['INDPRO_YOY']  = yoy_change('INDPRO', macro_df)
macro_df['MX_CPI_YOY']  = yoy_change('MX_CPI', macro_df)

# Curvas de interés
if 'DGS10' in macro_df.columns and 'DGS2' in macro_df.columns:
    macro_df['CURVE_10Y2Y'] = macro_df['DGS10'] - macro_df['DGS2']
if 'DGS10' in macro_df.columns and 'DGS3MO' in macro_df.columns:
    macro_df['CURVE_10Y3M'] = macro_df['DGS10'] - macro_df['DGS3MO']
if 'BAA' in macro_df.columns and 'DGS10' in macro_df.columns:
    macro_df['SPREAD_BAA10Y'] = macro_df['BAA'] - macro_df['DGS10']
if 'MX_10Y' in macro_df.columns and 'DGS10' in macro_df.columns:
    macro_df['SPREAD_MX_US_10Y'] = macro_df['MX_10Y'] - macro_df['DGS10']
if 'MX_10Y' in macro_df.columns and 'MX_3M' in macro_df.columns:
    macro_df['CURVE_MX_10Y3M'] = macro_df['MX_10Y'] - macro_df['MX_3M']

# Fortaleza del peso (inverso del USD/MXN)
if 'USDMXN' in prices_df.columns:
    prices_df['MXNUSD'] = 1.0 / prices_df['USDMXN']

print(f'✅ macro_df shape  : {macro_df.shape}')
print(f'   Columnas: {list(macro_df.columns)}')

✅ macro_df shape  : (3132, 36)
   Columnas: ['EFFR', 'DGS10', 'DGS2', 'DGS3MO', 'CPI', 'CORECPI', 'PCE', 'COREPCE', 'BREAKEVEN5Y', 'BREAKEVEN10Y', 'UNRATE', 'CLAIMS', 'RETAIL', 'REALPCE', 'INDPRO', 'SENTIMENT', 'ISM_PMI', 'NFCI', 'HY_SPREAD', 'BAA', 'M2', 'TWD', 'MX_CPI', 'MX_3M', 'MX_10Y', 'CPI_YOY', 'CORECPI_YOY', 'PCE_YOY', 'M2_YOY', 'INDPRO_YOY', 'MX_CPI_YOY', 'CURVE_10Y2Y', 'CURVE_10Y3M', 'SPREAD_BAA10Y', 'SPREAD_MX_US_10Y', 'CURVE_MX_10Y3M']


In [7]:
# ============================================================
# CELDA 7: FEATURE ENGINEERING
# ============================================================

feat = pd.DataFrame(index=bday_idx)

# ---------- MACRO FRED (niveles + cambios 21 bdays) ----------
macro_cols_use = [
    'EFFR','DGS10','DGS2','DGS3MO',
    'CURVE_10Y2Y','CURVE_10Y3M',
    'CPI_YOY','CORECPI_YOY','PCE_YOY',
    'BREAKEVEN5Y','BREAKEVEN10Y',
    'UNRATE','CLAIMS',
    'NFCI','HY_SPREAD','SPREAD_BAA10Y',
    'M2_YOY','INDPRO_YOY','SENTIMENT',
    'ISM_PMI',
    'MX_CPI_YOY','MX_3M','MX_10Y',
    'CURVE_MX_10Y3M','SPREAD_MX_US_10Y',
    'TWD',
]

for col in macro_cols_use:
    if col in macro_df.columns:
        feat[col] = macro_df[col]
        # Cambio 21 días hábiles
        feat[f'{col}_D21'] = macro_df[col].diff(21)

# ---------- PRECIOS DE MERCADO ----------
price_feat_keys = ['USDMXN','DXY','WTI','VIX','GLD','SLV']
for k in price_feat_keys:
    if k in prices_df.columns:
        feat[k] = prices_df[k]
        feat[f'{k}_D21'] = prices_df[k].pct_change(21)
        feat[f'{k}_D5']  = prices_df[k].pct_change(5)

# Fortaleza del peso
if 'MXNUSD' in prices_df.columns:
    feat['MXNUSD'] = prices_df['MXNUSD']

# Volatilidad realizada 21d de SPY como proxy de riesgo general
if 'SPY' in prices_df.columns:
    spy_ret = np.log(prices_df['SPY']).diff()
    feat['SPY_RVOL21'] = spy_ret.rolling(21).std() * np.sqrt(252)
    feat['SPY_RVOL63'] = spy_ret.rolling(63).std() * np.sqrt(252)

# Recortar a ventana de 5 años y limpiar
feat = feat[feat.index >= FOCUS_START].copy()
feat = feat.ffill().dropna(how='all')

print(f'✅ features_daily shape: {feat.shape}')
print(f'   {feat.shape[1]} features | {feat.shape[0]} días')
null_frac = feat.isnull().mean()
high_null = null_frac[null_frac > 0.3]
if not high_null.empty:
    print(f'   ⚠️  Features con >30% NaN: {list(high_null.index)}')

✅ features_daily shape: (1305, 73)
   73 features | 1305 días
   ⚠️  Features con >30% NaN: ['CLAIMS', 'CLAIMS_D21']


In [8]:
# ============================================================
# CELDA 8: TARGETS — FORWARD RETURNS 5 DÍAS
# ============================================================

# Retornos log diarios
log_prices = np.log(prices_df[ASSET_KEYS].ffill())
daily_ret  = log_prices.diff()

# Forward return 5 días: ln(P(t+5)/P(t)) — SIN lookahead
# IMPORTANTE: shift(-5) SOLO para el target, nunca para los features
fwd5 = log_prices.diff(5).shift(-5)  # ret del período [t, t+5]

# Recortar ambos a ventana de 5 años
daily_ret = daily_ret[daily_ret.index >= FOCUS_START]
fwd5      = fwd5[fwd5.index >= FOCUS_START]

# Dataset final: features + targets alineados
common_idx = feat.index.intersection(fwd5.index)
df_feat = feat.reindex(common_idx)
df_fwd5 = fwd5.reindex(common_idx)

print(f'✅ Forward returns 5d shape: {df_fwd5.shape}')
print(f'   Rango: {df_fwd5.index[0].date()} → {df_fwd5.index[-1].date()}')
valid_fwd = df_fwd5.notna().sum()
print(f'   Observaciones válidas por activo:')
for k in ASSET_KEYS:
    if k in df_fwd5.columns:
        print(f'     {FULL_NAME[k]:45s}: {valid_fwd[k]}')

✅ Forward returns 5d shape: (1305, 15)
   Rango: 2021-03-11 → 2026-03-11
   Observaciones válidas por activo:
     S&P 500 (ETF SPY)                            : 1300
     Nasdaq 100 (ETF QQQ)                         : 1300
     Oro (ETF GLD)                                : 1300
     Plata (ETF SLV)                              : 1300
     Sector Comunicaciones (XLC)                  : 1300
     Sector Consumo Discrecional (XLY)            : 1300
     Sector Consumo Básico (XLP)                  : 1300
     Sector Energía (XLE)                         : 1300
     Sector Financiero (XLF)                      : 1300
     Sector Salud (XLV)                           : 1300
     Sector Industrial (XLI)                      : 1300
     Sector Materiales (XLB)                      : 1300
     Sector Bienes Raíces (XLRE)                  : 1300
     Sector Tecnología (XLK)                      : 1300
     Sector Servicios Públicos (XLU)              : 1300


In [9]:
# ============================================================
# CELDA 9: CORRELACIONES FEATURES → FORWARD RETURNS 5D
# ============================================================

# Correlaciones Pearson de cada feature con cada forward return 5d
corr_matrix = pd.DataFrame(index=df_feat.columns, columns=df_fwd5.columns, dtype=float)

for feat_col in df_feat.columns:
    for asset_col in df_fwd5.columns:
        valid = df_feat[feat_col].notna() & df_fwd5[asset_col].notna()
        if valid.sum() > 50:
            corr_matrix.loc[feat_col, asset_col] = df_feat.loc[valid, feat_col].corr(
                df_fwd5.loc[valid, asset_col])

corr_matrix = corr_matrix.astype(float)

# Stack para ranking
corr_stack = corr_matrix.stack().reset_index()
corr_stack.columns = ['Feature', 'Activo', 'Correlacion']
corr_stack['Feature_Nombre'] = corr_stack['Feature']
corr_stack['Activo_Nombre'] = corr_stack['Activo'].map(FULL_NAME)
corr_stack = corr_stack.dropna(subset=['Correlacion'])
corr_stack = corr_stack.sort_values('Correlacion', ascending=False)

top_pos = corr_stack.head(25).copy()
top_neg = corr_stack.tail(25).copy()

# Min/Max por indicador
minmax_by_ind = corr_matrix.apply(
    lambda row: pd.Series({
        'Corr_Min': row.min(),
        'Activo_Min': FULL_NAME.get(row.idxmin(), row.idxmin()) if pd.notna(row.min()) else '',
        'Corr_Max': row.max(),
        'Activo_Max': FULL_NAME.get(row.idxmax(), row.idxmax()) if pd.notna(row.max()) else '',
    }), axis=1
).reset_index()
minmax_by_ind.columns = ['Indicador','Corr_Min','Activo_Min','Corr_Max','Activo_Max']

print(f'✅ Correlaciones calculadas: {corr_matrix.shape}')
print(f'\n🔝 Top 5 correlaciones POSITIVAS:')
print(top_pos[['Feature','Activo_Nombre','Correlacion']].head())
print(f'\n🔻 Top 5 correlaciones NEGATIVAS:')
print(top_neg[['Feature','Activo_Nombre','Correlacion']].tail())

✅ Correlaciones calculadas: (73, 15)

🔝 Top 5 correlaciones POSITIVAS:
    Feature                Activo_Nombre  Correlacion
4      EFFR  Sector Comunicaciones (XLC)     0.169079
94   DGS3MO  Sector Comunicaciones (XLC)     0.148989
932     GLD                Oro (ETF GLD)     0.129471
604   MX_3M  Sector Comunicaciones (XLC)     0.126856
895     VIX      Sector Industrial (XLI)     0.119408

🔻 Top 5 correlaciones NEGATIVAS:
          Feature                Activo_Nombre  Correlacion
871        WTI_D5         Nasdaq 100 (ETF QQQ)    -0.197500
881        WTI_D5      Sector Materiales (XLB)    -0.205039
870        WTI_D5            S&P 500 (ETF SPY)    -0.205631
304  BREAKEVEN10Y  Sector Comunicaciones (XLC)    -0.245627
274   BREAKEVEN5Y  Sector Comunicaciones (XLC)    -0.251325


In [10]:
# ============================================================
# CELDA 10: VOLATILIDAD POR SECTOR
# ============================================================

SECTOR_KEYS = ['XLC','XLY','XLP','XLE','XLF','XLV','XLI','XLB','XLRE','XLK','XLU']
sector_keys_avail = [k for k in SECTOR_KEYS if k in prices_df.columns]

# Retornos log diarios de sectores
sector_prices = prices_df[sector_keys_avail].ffill()
sector_ret    = np.log(sector_prices).diff()
sector_ret    = sector_ret[sector_ret.index >= FOCUS_START]

# Volatilidad rolling anualizada
vol21  = sector_ret.rolling(21).std()  * np.sqrt(252) * 100  # en %
vol63  = sector_ret.rolling(63).std()  * np.sqrt(252) * 100
vol252 = sector_ret.rolling(252).std() * np.sqrt(252) * 100

# Volatilidad actual (último día disponible)
vol_current = pd.DataFrame({
    'Sector': [FULL_NAME.get(k, k) for k in sector_keys_avail],
    'Vol_21d_%':  vol21.iloc[-1][sector_keys_avail].values,
    'Vol_63d_%':  vol63.iloc[-1][sector_keys_avail].values,
    'Vol_252d_%': vol252.iloc[-1][sector_keys_avail].values,
}).set_index('Sector')

vol_current['Ratio_Vol21_vs_252'] = vol_current['Vol_21d_%'] / vol_current['Vol_252d_%']
vol_current['Regimen'] = vol_current['Ratio_Vol21_vs_252'].apply(
    lambda x: '🔴 ALTA VOLATILIDAD' if x > 1.3 else ('🟢 BAJA VOLATILIDAD' if x < 0.7 else '🟡 NORMAL')
)

vol_ranking = vol_current.sort_values('Vol_21d_%', ascending=False)

print('📊 Volatilidad actual por sector (anualizada):')
print(vol_ranking[['Vol_21d_%','Vol_63d_%','Regimen']].to_string())

📊 Volatilidad actual por sector (anualizada):
                                   Vol_21d_%  Vol_63d_%             Regimen
Sector                                                                     
Sector Tecnología (XLK)            20.756132  21.352388            🟡 NORMAL
Sector Financiero (XLF)            19.593276  16.282086            🟡 NORMAL
Sector Energía (XLE)               18.010725  21.386598            🟡 NORMAL
Sector Materiales (XLB)            16.320364  17.589315            🟡 NORMAL
Sector Industrial (XLI)            15.775711  15.575827            🟡 NORMAL
Sector Consumo Discrecional (XLY)  15.731747  15.329024  🟢 BAJA VOLATILIDAD
Sector Servicios Públicos (XLU)    15.505301  13.990571            🟡 NORMAL
Sector Consumo Básico (XLP)        15.090008  13.458036            🟡 NORMAL
Sector Salud (XLV)                 13.663635  13.315005            🟡 NORMAL
Sector Comunicaciones (XLC)        11.714519  11.798104  🟢 BAJA VOLATILIDAD
Sector Bienes Raíces (XLRE)        10.7377

In [11]:
# ============================================================
# CELDA 11: kNN — DÍAS PARECIDOS
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Features de estado para kNN
STATE_FEATURES = [
    'EFFR','DGS10','DGS2','CURVE_10Y2Y','CURVE_10Y3M',
    'HY_SPREAD','NFCI','SPREAD_BAA10Y',
    'USDMXN','DXY','WTI','VIX',
    'CLAIMS','UNRATE','BREAKEVEN5Y',
    'SPY_RVOL21',
]
state_feats_avail = [f for f in STATE_FEATURES if f in df_feat.columns]

# Dataset para kNN: solo filas sin NaN en features de estado
knn_data = df_feat[state_feats_avail].copy()
valid_rows = knn_data.dropna()
fwd5_knn   = df_fwd5.reindex(valid_rows.index)

# Z-score
scaler = StandardScaler()
X_scaled = scaler.fit_transform(valid_rows)

# Estado actual (último punto disponible sin NaN en features de estado)
last_valid_idx = valid_rows.index[-1]
x_today = scaler.transform(valid_rows.iloc[[-1]])

# kNN
K = min(K_NEIGHBORS, len(valid_rows) - 1)
nn = NearestNeighbors(n_neighbors=K + 1, metric='euclidean')
nn.fit(X_scaled)
distances, indices = nn.kneighbors(x_today)

# Excluir el propio punto (índice 0 si está incluido)
neighbor_idx = indices[0][1:]  # Siempre excluir el primer vecino (el propio punto)

knn_results = {}
for asset in ASSET_KEYS:
    if asset not in fwd5_knn.columns:
        continue
    neighbor_returns = fwd5_knn.iloc[neighbor_idx][asset].dropna().values
    if len(neighbor_returns) < 10:
        continue
    knn_results[asset] = {
        'Nombre': FULL_NAME.get(asset, asset),
        'P_subir_%': (neighbor_returns > 0).mean() * 100,
        'Ret_Esperado_%': np.mean(neighbor_returns) * 100,
        'VaR5_%': np.percentile(neighbor_returns, 5) * 100,
        'P10_%': np.percentile(neighbor_returns, 10) * 100,
        'P50_%': np.percentile(neighbor_returns, 50) * 100,
        'P90_%': np.percentile(neighbor_returns, 90) * 100,
        'P95_%': np.percentile(neighbor_returns, 95) * 100,
        'N_vecinos': len(neighbor_returns),
    }

knn_df = pd.DataFrame(knn_results).T.reset_index(names='Ticker')
print(f'✅ kNN completado con K={K} vecinos | {last_valid_idx.date()}')
print(knn_df[['Nombre','P_subir_%','Ret_Esperado_%','VaR5_%']].to_string())

ValueError: Found array with 0 sample(s) (shape=(0, 16)) while a minimum of 1 is required by StandardScaler.

In [ ]:
# ============================================================
# CELDA 12: REGRESIÓN LOGÍSTICA CON EXPLICACIÓN DE DRIVERS
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Features para logistic: state features + cambios 21d
LOGIT_FEATURES = state_feats_avail + [
    f for f in df_feat.columns if f.endswith('_D21') and f in df_feat.columns
]

logit_data = df_feat[LOGIT_FEATURES].copy().dropna()
logit_results = {}
logit_drivers = {}  # drivers por activo

# Medianas históricas de cada feature (para comparación en conclusiones)
feat_medians = df_feat[LOGIT_FEATURES].median()
feat_current = df_feat[LOGIT_FEATURES].ffill().iloc[-1]

for asset in ASSET_KEYS:
    if asset not in df_fwd5.columns:
        continue
    target = df_fwd5[asset].reindex(logit_data.index)
    valid  = target.notna()
    X_     = logit_data[valid].values
    y_     = (target[valid] > 0).astype(int).values
    if y_.sum() < 20 or (1 - y_).sum() < 20:
        continue

    # Modelo logístico con regularización
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('logit', LogisticRegression(C=0.1, max_iter=1000, random_state=SEED))
    ])
    pipe.fit(X_, y_)

    # Predecir para el estado actual
    x_now = feat_current[LOGIT_FEATURES].values.reshape(1, -1)
    # Manejar NaN en estado actual
    x_now_filled = np.where(np.isnan(x_now), feat_medians[LOGIT_FEATURES].values, x_now)

    prob_up = pipe.predict_proba(x_now_filled)[0][1]

    # Coeficientes estandarizados
    coefs = pipe.named_steps['logit'].coef_[0]
    driver_df = pd.DataFrame({
        'Feature': LOGIT_FEATURES,
        'Coeficiente': coefs,
        'Valor_Actual': feat_current[LOGIT_FEATURES].values,
        'Mediana_5y': feat_medians[LOGIT_FEATURES].values,
    })
    driver_df['Impacto'] = driver_df['Coeficiente'] * (
        driver_df['Valor_Actual'] - driver_df['Mediana_5y']
    )
    driver_df['Direccion'] = driver_df['Impacto'].apply(
        lambda x: '↑ POSITIVO' if x > 0 else '↓ NEGATIVO')
    driver_df = driver_df.sort_values('Impacto', key=abs, ascending=False)

    logit_results[asset] = {
        'Nombre': FULL_NAME.get(asset, asset),
        'P_subir_%': prob_up * 100,
        'N_obs': int(valid.sum()),
    }
    logit_drivers[asset] = driver_df.head(10).copy()

logit_df = pd.DataFrame(logit_results).T.reset_index(names='Ticker')
print(f'✅ Logistic completado para {len(logit_results)} activos')
print(logit_df[['Nombre','P_subir_%']].to_string())

In [ ]:
# ============================================================
# CELDA 13: MONTE CARLO — VAR(1) + SHOCKS CORRELACIONADOS
#           batching ≥ 1,000,000 simulaciones
# ============================================================
import warnings
warnings.filterwarnings('ignore')

# ---- 13.1 VAR(1) sobre endógenos ----
endog_avail = [k for k in ENDOG_KEYS if k in prices_df.columns]
endog_prices = prices_df[endog_avail].ffill()
endog_ret    = np.log(endog_prices).diff().dropna()
endog_ret    = endog_ret[endog_ret.index >= FOCUS_START]

N_ENDOG = len(endog_avail)
var_ok = False
var_fallback_msg = ''

if len(endog_ret) > 100 and N_ENDOG >= 2:
    try:
        from statsmodels.tsa.vector_ar.var_model import VAR
        var_model = VAR(endog_ret)
        var_fit   = var_model.fit(maxlags=1, ic='aic', verbose=False)
        # Parámetros VAR(1)
        A1_raw   = var_fit.coefs  # shape (lags, n, n)
        A1       = A1_raw[0]      # (n, n)
        intercept= var_fit.intercept  # (n,)
        resid    = var_fit.resid      # (T-1, n)
        Sigma_endog = np.cov(resid.T) # covarianza residuos VAR
        # Estado inicial: último retorno disponible
        last_endog = endog_ret.iloc[-1].values
        var_ok = True
        print(f'✅ VAR(1) estimado sobre: {endog_avail}')
        print(f'   Observaciones: {len(endog_ret)}')
    except Exception as e:
        var_fallback_msg = f'VAR(1) falló ({e}). Usando covarianza histórica simple.'
        print(f'⚠️  {var_fallback_msg}')
else:
    var_fallback_msg = 'Datos insuficientes para VAR(1). Usando covarianza histórica simple.'
    print(f'⚠️  {var_fallback_msg}')

# ---- Fallback: multivariate normal ----
if not var_ok:
    A1 = np.zeros((N_ENDOG, N_ENDOG))
    intercept = endog_ret.mean().values if N_ENDOG > 0 else np.zeros(1)
    Sigma_endog = np.cov(endog_ret.T) if N_ENDOG > 1 else np.array([[endog_ret.var().values[0]]])
    last_endog = endog_ret.iloc[-1].values if len(endog_ret) > 0 else np.zeros(N_ENDOG)

# ---- 13.2 OLS: retornos de activos ~ endógenos ----
from sklearn.linear_model import LinearRegression

# Alinear retornos de activos con endógenos
asset_ret_daily = np.log(prices_df[ASSET_KEYS].ffill()).diff()
asset_ret_daily = asset_ret_daily[asset_ret_daily.index >= FOCUS_START]
endog_ret_algn  = endog_ret.reindex(asset_ret_daily.index)

ols_betas     = {}  # asset -> beta vector (n_endog,)
ols_intercept = {}  # asset -> scalar
ols_resid_cov = None
asset_keys_ols = []

ols_resid_list = []

for asset in ASSET_KEYS:
    if asset not in asset_ret_daily.columns:
        continue
    y_ = asset_ret_daily[asset]
    X_ = endog_ret_algn[endog_avail]
    valid = y_.notna() & X_.notna().all(axis=1)
    if valid.sum() < 50:
        continue
    ols = LinearRegression()
    ols.fit(X_[valid], y_[valid])
    ols_betas[asset]     = ols.coef_
    ols_intercept[asset] = ols.intercept_
    res = y_[valid].values - ols.predict(X_[valid])
    ols_resid_list.append(res)
    asset_keys_ols.append(asset)

N_ASSETS_MC = len(asset_keys_ols)
print(f'✅ OLS estimado para {N_ASSETS_MC} activos')

# Covarianza de residuos entre activos (para shocks correlacionados)
min_len = min(len(r) for r in ols_resid_list) if ols_resid_list else 0
if min_len > 20:
    resid_mat = np.column_stack([r[-min_len:] for r in ols_resid_list])  # (T, N_assets)
    Sigma_asset = np.cov(resid_mat.T)
    # Cholesky para Sigma_asset
    try:
        L_asset = np.linalg.cholesky(Sigma_asset + 1e-8 * np.eye(N_ASSETS_MC))
    except np.linalg.LinAlgError:
        L_asset = np.diag(np.sqrt(np.maximum(np.diag(Sigma_asset), 1e-12)))
else:
    Sigma_asset = np.eye(N_ASSETS_MC) * 1e-4
    L_asset = np.diag(np.sqrt(np.diag(Sigma_asset)))

# Cholesky para VAR
try:
    if N_ENDOG == 1:
        L_endog = np.array([[np.sqrt(max(Sigma_endog.flat[0], 1e-12))]])
    else:
        L_endog = np.linalg.cholesky(Sigma_endog + 1e-8 * np.eye(N_ENDOG))
except np.linalg.LinAlgError:
    L_endog = np.diag(np.sqrt(np.maximum(np.diag(Sigma_endog), 1e-12)))

print(f'   L_endog shape  : {L_endog.shape}')
print(f'   L_asset shape  : {L_asset.shape}')

In [ ]:
# ============================================================
# CELDA 13b: MONTE CARLO — LOOP DE BATCHING
# ============================================================

N_BATCHES = N_SIM // BATCH_SIZE + (1 if N_SIM % BATCH_SIZE else 0)
print(f'🎲 Monte Carlo: {N_SIM:,} sims | {N_BATCHES} batches x {BATCH_SIZE:,}')

# Acumuladores
count_up  = np.zeros(N_ASSETS_MC, dtype=np.int64)
sum_ret   = np.zeros(N_ASSETS_MC, dtype=np.float64)
# Para percentiles: guardar hasta 500k retornos finales por activo (float32)
MAX_STORE = min(N_SIM, 500_000)
ret_store = np.empty((MAX_STORE, N_ASSETS_MC), dtype=np.float32)
stored    = 0
total_sim = 0

rng = np.random.default_rng(SEED)

for batch_i in range(N_BATCHES):
    bs = min(BATCH_SIZE, N_SIM - total_sim)
    if bs <= 0:
        break

    # Simular HORIZON_BDAYS pasos diarios
    # Retornos acumulados de activos (bs, N_ASSETS_MC)
    cum_asset_ret = np.zeros((bs, N_ASSETS_MC), dtype=np.float64)

    endog_state = np.tile(last_endog, (bs, 1))  # (bs, N_ENDOG)

    for t in range(HORIZON_BDAYS):
        # Shocks VAR (bs, N_ENDOG)
        z_endog = rng.standard_normal((bs, N_ENDOG))
        shock_endog = z_endog @ L_endog.T

        # Nuevo estado endógeno: e(t+1) = intercept + A1 @ e(t) + shock
        endog_state = intercept + endog_state @ A1.T + shock_endog

        # Retornos de activos (bs, N_ASSETS_MC)
        # ret_asset = ols_intercept + endog_state @ betas + resid_shock
        z_asset = rng.standard_normal((bs, N_ASSETS_MC))
        shock_asset = z_asset @ L_asset.T

        for i, asset in enumerate(asset_keys_ols):
            # Componente VAR explicada
            endog_contribution = endog_state @ ols_betas[asset]
            day_ret = ols_intercept[asset] + endog_contribution + shock_asset[:, i]
            cum_asset_ret[:, i] += day_ret

    # Acumular estadísticas
    count_up  += (cum_asset_ret > 0).sum(axis=0)
    sum_ret   += cum_asset_ret.sum(axis=0)

    # Guardar retornos para percentiles (hasta MAX_STORE)
    space = MAX_STORE - stored
    if space > 0:
        n_store = min(bs, space)
        ret_store[stored:stored + n_store] = cum_asset_ret[:n_store].astype(np.float32)
        stored += n_store

    total_sim += bs
    if (batch_i + 1) % 5 == 0 or batch_i == N_BATCHES - 1:
        print(f'  Batch {batch_i+1}/{N_BATCHES} completado | {total_sim:,} sims')

print(f'\n✅ Monte Carlo completado: {total_sim:,} simulaciones')

# Calcular resultados finales
ret_store_valid = ret_store[:stored]

mc_results = {}
for i, asset in enumerate(asset_keys_ols):
    p_up   = count_up[i] / total_sim * 100
    e_ret  = sum_ret[i]  / total_sim * 100
    col_ret = ret_store_valid[:, i] * 100  # en %
    mc_results[asset] = {
        'Nombre':         FULL_NAME.get(asset, asset),
        'P_subir_%':      p_up,
        'Ret_Esperado_%': e_ret,
        'VaR5_%':         np.percentile(col_ret, 5),
        'P10_%':          np.percentile(col_ret, 10),
        'P50_%':          np.percentile(col_ret, 50),
        'P90_%':          np.percentile(col_ret, 90),
        'P95_%':          np.percentile(col_ret, 95),
    }

mc_df = pd.DataFrame(mc_results).T.reset_index(names='Ticker')
print(mc_df[['Nombre','P_subir_%','Ret_Esperado_%','VaR5_%']].to_string())

In [ ]:
# ============================================================
# CELDA 14: RANKING FINAL COMBINADO
# ============================================================

# Pesos de combinación
W_MC     = 0.50
W_LOGIT  = 0.30
W_KNN    = 0.20

ranking_rows = []

for asset in ASSET_KEYS:
    row = {'Ticker': asset, 'Nombre': FULL_NAME.get(asset, asset)}

    p_mc    = mc_results.get(asset, {}).get('P_subir_%', np.nan)
    p_logit = logit_results.get(asset, {}).get('P_subir_%', np.nan)
    p_knn   = knn_results.get(asset, {}).get('P_subir_%', np.nan)

    row['P_MC_%']    = p_mc
    row['P_Logit_%'] = p_logit
    row['P_kNN_%']   = p_knn

    # Puntaje combinado (solo con los que tienen dato)
    vals  = [v for v in [(p_mc, W_MC), (p_logit, W_LOGIT), (p_knn, W_KNN)] if not np.isnan(v[0])]
    if vals:
        total_w = sum(w for _, w in vals)
        row['P_Combinada_%'] = sum(v * w for v, w in vals) / total_w
    else:
        row['P_Combinada_%'] = np.nan

    # VaR y retorno esperado de Monte Carlo
    row['Ret_Esperado_MC_%'] = mc_results.get(asset, {}).get('Ret_Esperado_%', np.nan)
    row['VaR5_MC_%']         = mc_results.get(asset, {}).get('VaR5_%', np.nan)
    row['P10_MC_%']          = mc_results.get(asset, {}).get('P10_%', np.nan)
    row['P50_MC_%']          = mc_results.get(asset, {}).get('P50_%', np.nan)
    row['P90_MC_%']          = mc_results.get(asset, {}).get('P90_%', np.nan)

    # Convergencia de modelos
    ps = [p for p in [p_mc, p_logit, p_knn] if not np.isnan(p)]
    if len(ps) >= 2:
        row['Rango_Modelos_%'] = max(ps) - min(ps)
        row['Convergencia'] = '✅ Coinciden' if row['Rango_Modelos_%'] < 10 else '⚠️ Divergen'
    else:
        row['Rango_Modelos_%'] = np.nan
        row['Convergencia'] = 'N/A'

    ranking_rows.append(row)

ranking_df = pd.DataFrame(ranking_rows)
ranking_df = ranking_df.sort_values('P_Combinada_%', ascending=False).reset_index(drop=True)
ranking_df['Rank'] = ranking_df.index + 1

top3_up   = ranking_df.head(3)
bottom3   = ranking_df.tail(3)

print('🏆 TOP 3 MÁS PROBABLE QUE SUBA:')
print(top3_up[['Nombre','P_Combinada_%','P_MC_%','P_Logit_%','P_kNN_%']].to_string())
print('\n🔻 BOTTOM 3 MÁS PROBABLE QUE BAJE:')
print(bottom3[['Nombre','P_Combinada_%','P_MC_%','P_Logit_%','P_kNN_%']].to_string())

In [ ]:
# ============================================================
# CELDA 15: HOJA DEFINICIONES
# ============================================================

DEFINICIONES = {
    'EFFR':              'Tasa de fondos federales efectiva (Federal Funds Rate). Tasa a la que bancos se prestan reservas overnight. Principal instrumento de política monetaria de la Fed.',
    'DGS10':             'Rendimiento del bono del Tesoro EUA a 10 años. Referencia global de tasas de largo plazo; sube con expectativas de inflación o crecimiento.',
    'DGS2':              'Rendimiento del bono del Tesoro EUA a 2 años. Refleja expectativas de política monetaria a corto-mediano plazo.',
    'DGS3MO':            'Rendimiento de la Letra del Tesoro a 3 meses. Proxy de la tasa libre de riesgo de muy corto plazo.',
    'CURVE_10Y2Y':       'Spread de curva 10Y-2Y (diferencia de rendimientos). Positivo=curva normal; negativo=curva invertida (señal histórica de recesión).',
    'CURVE_10Y3M':       'Spread de curva 10Y-3M. Predictor de recesión con mayor poder estadístico histórico que 10Y-2Y según la Fed de Nueva York.',
    'CPI_YOY':           'Inflación CPI año contra año (Consumer Price Index). Mide cambio en precio de una canasta de bienes y servicios para consumidores urbanos EUA.',
    'CORECPI_YOY':       'Inflación Core CPI (excluye alimentos y energía). Más estable y seguida por la Fed para decisiones de política monetaria.',
    'PCE_YOY':           'Índice de Precios del Gasto en Consumo Personal. Indicador de inflación preferido de la Fed para su objetivo del 2%.',
    'COREPCE':           'Core PCE (excluye alimentos y energía). El indicador de inflación MÁS importante para la Fed.',
    'BREAKEVEN5Y':       'Tasa breakeven de inflación a 5 años (TIPS vs nominales). Mide expectativas de inflación del mercado para los próximos 5 años.',
    'BREAKEVEN10Y':      'Tasa breakeven de inflación a 10 años. Expectativa de inflación promedio a 10 años implícita en precios de mercado.',
    'UNRATE':            'Tasa de desempleo EUA (Bureau of Labor Statistics). Indicador clave de la salud del mercado laboral.',
    'CLAIMS':            'Solicitudes iniciales de desempleo semanales (Initial Jobless Claims). Indicador de alta frecuencia y leading de actividad laboral.',
    'RETAIL':            'Ventas al menudeo EUA (Retail Sales). Proxy del gasto del consumidor, que representa ~70% del PIB EUA.',
    'REALPCE':           'Gasto Real en Consumo Personal. Medida ajustada por inflación del consumo; determinante directo del PIB.',
    'INDPRO':            'Índice de Producción Industrial. Mide output de manufactura, minería y servicios públicos; indicador coincidente del ciclo económico.',
    'SENTIMENT':         'Índice de Confianza del Consumidor (U. of Michigan). Leading indicator que anticipa gasto futuro del consumidor.',
    'ISM_PMI':           'ISM PMI Manufacturero. Difusion index: >50=expansión, <50=contracción. Leading indicator del ciclo industrial.',
    'NFCI':              'Índice Nacional de Condiciones Financieras (Chicago Fed). Valores negativos=condiciones laxas; positivos=condiciones restrictivas.',
    'HY_SPREAD':         'Spread de crédito High Yield (bonos basura) vs Tesoros. Mide aversión al riesgo; spreads altos=estrés financiero.',
    'SPREAD_BAA10Y':     'Spread Baa corporativo vs bono 10Y. Indicador de riesgo de crédito corporativo; sube en recesiones.',
    'M2_YOY':            'Crecimiento año vs año del agregado monetario M2 (depósitos, cuentas de ahorro). Indicador de liquidez del sistema.',
    'INDPRO_YOY':        'Crecimiento año vs año de la Producción Industrial. Sintetiza expansión/contracción del sector real.',
    'TWD':               'Índice del dólar ponderado por comercio (Trade-Weighted Dollar Index, FRED). Proxy oficial de la fortaleza del dólar.',
    'MX_CPI_YOY':        'Inflación México año vs año (OCDE/FRED). Afecta política monetaria de Banxico y el tipo de cambio USD/MXN.',
    'MX_3M':             'Tasa de interés de corto plazo México a 3 meses. Proxy de la tasa de referencia de Banxico.',
    'MX_10Y':            'Rendimiento del bono gubernamental México a 10 años (M10). Referencia para deuda pública y spreads soberanos.',
    'CURVE_MX_10Y3M':    'Curva de rendimientos México 10Y-3M. Pendiente positiva=expectativas de crecimiento; invertida=estrés.',
    'SPREAD_MX_US_10Y':  'Spread soberano México vs EUA a 10 años. Prima de riesgo país; sube con riesgo político o fiscal.',
    'USDMXN':            'Tipo de cambio USD/MXN (pesos mexicanos por dólar). Central para importaciones, deuda en dólares y costos de PYP.',
    'DXY':               'Índice del Dólar (ICE). Canasta de 6 divisas (EUR 57.6%, JPY 13.6%, GBP 11.9%, CAD 9.1%, SEK 4.2%, CHF 3.6%). Fortaleza del dólar.',
    'WTI':               'Petróleo West Texas Intermediate. Referencia de precio del crudo. Afecta inflación, costos logísticos y sectores de energía.',
    'VIX':               'Índice de Volatilidad del S&P 500 (CBOE). Mide volatilidad implícita a 30 días; apodado "indicador del miedo".',
    'MXNUSD':            'Fortaleza del peso mexicano (inverso del USD/MXN = MXN/USD). Sube cuando el peso se aprecia.',
    'SPY_RVOL21':        'Volatilidad realizada del S&P 500 a 21 días hábiles, anualizada. Proxy del régimen de riesgo actual del mercado.',
}

defs_rows = []
for feat_name, descrip in DEFINICIONES.items():
    row_d = {'Indicador': feat_name, 'Descripcion': descrip}
    if feat_name in minmax_by_ind['Indicador'].values:
        m = minmax_by_ind[minmax_by_ind['Indicador'] == feat_name].iloc[0]
        row_d['Corr_Min'] = m['Corr_Min']
        row_d['Activo_Min'] = m['Activo_Min']
        row_d['Corr_Max'] = m['Corr_Max']
        row_d['Activo_Max'] = m['Activo_Max']
    defs_rows.append(row_d)

defs_df = pd.DataFrame(defs_rows)
print(f'✅ Hoja DEFINICIONES: {len(defs_df)} indicadores definidos')

In [ ]:
# ============================================================
# CELDA 16: READOUT HOY + CONCLUSIONES HUMANAS EXTENSAS
# ============================================================

# ---- READOUT HOY ----
readout_rows = []
for col in df_feat.columns:
    series = df_feat[col].dropna()
    if len(series) < 10:
        continue
    val_now   = series.iloc[-1]
    med_5y    = series.median()
    pct_rank  = (series < val_now).mean() * 100  # percentil actual dentro de 5 años
    readout_rows.append({
        'Indicador':    col,
        'Descripcion':  DEFINICIONES.get(col, col),
        'Valor_Actual': val_now,
        'Mediana_5y':   med_5y,
        'Percentil_5y': pct_rank,
        'vs_Mediana':   'ALTO' if val_now > med_5y else 'BAJO',
    })

readout_df = pd.DataFrame(readout_rows)

def get_val(indicator):
    """Obtiene valor actual de un indicador."""
    if indicator in df_feat.columns:
        s = df_feat[indicator].dropna()
        if len(s) > 0:
            return s.iloc[-1]
    return np.nan

def get_pct(indicator):
    """Percentil actual dentro de 5 años (0-100)."""
    if indicator in df_feat.columns:
        s = df_feat[indicator].dropna()
        if len(s) > 0:
            v = s.iloc[-1]
            return (s < v).mean() * 100
    return np.nan

def fmt(v, decimals=2):
    if pd.isna(v):
        return 'N/D'
    return f'{v:.{decimals}f}'

def fmt_pct(v):
    if pd.isna(v):
        return 'N/D'
    return f'{v:.1f}%'

# Construir texto de conclusión
fecha_hoy = TODAY.strftime('%d de %B de %Y')

# ---- Rankings para texto ----
top3_names  = top3_up['Nombre'].tolist()
bot3_names  = bottom3['Nombre'].tolist()

def build_asset_paragraph(asset_key):
    """Genera párrafo de análisis para un activo."""
    name    = FULL_NAME.get(asset_key, asset_key)
    mc      = mc_results.get(asset_key, {})
    lgt     = logit_results.get(asset_key, {})
    knn_r   = knn_results.get(asset_key, {})
    rank_r  = ranking_df[ranking_df['Ticker'] == asset_key]

    p_mc    = mc.get('P_subir_%', np.nan)
    p_lgt   = lgt.get('P_subir_%', np.nan)
    p_knn   = knn_r.get('P_subir_%', np.nan)
    p_comb  = rank_r['P_Combinada_%'].values[0] if len(rank_r) else np.nan
    e_ret   = mc.get('Ret_Esperado_%', np.nan)
    var5    = mc.get('VaR5_%', np.nan)
    p10     = mc.get('P10_%', np.nan)
    p50     = mc.get('P50_%', np.nan)
    p90     = mc.get('P90_%', np.nan)
    conv    = rank_r['Convergencia'].values[0] if len(rank_r) else 'N/A'

    # Drivers logísticos
    drivers = logit_drivers.get(asset_key, pd.DataFrame())
    driver_text = ''
    if not drivers.empty:
        for _, dr in drivers.head(5).iterrows():
            f   = dr['Feature']
            va  = dr['Valor_Actual']
            med = dr['Mediana_5y']
            imp = dr['Impacto']
            desc = DEFINICIONES.get(f, f)
            direccion = 'por encima' if va > med else 'por debajo'
            driver_text += (f'\n       • {desc[:60]}... '
                           f'(valor actual: {fmt(va, 3)}, mediana 5y: {fmt(med, 3)}, '
                           f'{direccion} de la mediana) — impacto {'POSITIVO' if imp > 0 else 'NEGATIVO'} en la probabilidad')

    texto = f"""
--- {name.upper()} ---
Probabilidad de subir en los próximos 5 días hábiles:
   • Monte Carlo ({N_SIM:,} simulaciones): {fmt_pct(p_mc)}
   • Modelo Logístico (regresión penalizada): {fmt_pct(p_lgt)}
   • kNN Días Parecidos (K={K} vecinos): {fmt_pct(p_knn)}
   → PROBABILIDAD COMBINADA: {fmt_pct(p_comb)}
   → Convergencia de modelos: {conv}

Retorno esperado (Monte Carlo, 5 días):
   • Promedio: {fmt_pct(e_ret)}
   • Escenario pesimista P10: {fmt_pct(p10)}
   • Mediana P50: {fmt_pct(p50)}
   • Escenario optimista P90: {fmt_pct(p90)}
   • VaR 5% (pérdida máxima esperada 95% del tiempo): {fmt_pct(var5)}

Principales drivers (según modelo logístico):{driver_text if driver_text else chr(10) + '       • No disponible.'}
"""
    return texto

# Macro conditions text
macro_text = f"""
=================================================================
  ESTUDIO MACRO-FINANCIERO 5 DÍAS — EUA + MÉXICO
  Fecha de análisis: {fecha_hoy}
  Monte Carlo: {N_SIM:,} simulaciones | VAR(1) + Cholesky
=================================================================

SECCIÓN 1: CONDICIONES MACRO ACTUALES
--------------------------------------

TASAS DE INTERÉS (EUA):
  • Tasa de fondos federales (EFFR): {fmt(get_val('EFFR'))}% — percentil {fmt(get_pct('EFFR'), 0)}/100 en 5 años
  • Bono 10 años (DGS10): {fmt(get_val('DGS10'))}% — percentil {fmt(get_pct('DGS10'), 0)}/100
  • Bono 2 años (DGS2): {fmt(get_val('DGS2'))}% — percentil {fmt(get_pct('DGS2'), 0)}/100
  • Curva 10Y-2Y: {fmt(get_val('CURVE_10Y2Y'))} pp — {'INVERTIDA (señal recesión)' if (not pd.isna(get_val('CURVE_10Y2Y')) and get_val('CURVE_10Y2Y') < 0) else 'NORMAL'}
  • Curva 10Y-3M: {fmt(get_val('CURVE_10Y3M'))} pp — {'INVERTIDA' if (not pd.isna(get_val('CURVE_10Y3M')) and get_val('CURVE_10Y3M') < 0) else 'NORMAL'}

INFLACIÓN (EUA):
  • CPI YoY: {fmt(get_val('CPI_YOY'))}% — percentil {fmt(get_pct('CPI_YOY'), 0)}/100
  • Core CPI YoY: {fmt(get_val('CORECPI_YOY'))}%
  • PCE YoY: {fmt(get_val('PCE_YOY'))}%
  • Breakeven inflación 5Y: {fmt(get_val('BREAKEVEN5Y'))}% (expectativa del mercado)

MERCADO LABORAL Y ACTIVIDAD (EUA):
  • Tasa de desempleo: {fmt(get_val('UNRATE'))}% — percentil {fmt(get_pct('UNRATE'), 0)}/100
  • Solicitudes iniciales de desempleo (semanal): {fmt(get_val('CLAIMS'), 0)} — percentil {fmt(get_pct('CLAIMS'), 0)}/100
  • Confianza del consumidor (U. Michigan): {fmt(get_val('SENTIMENT'), 1)} — percentil {fmt(get_pct('SENTIMENT'), 0)}/100
  • Producción Industrial YoY: {fmt(get_val('INDPRO_YOY'))}%

CONDICIONES FINANCIERAS Y RIESGO (EUA):
  • NFCI (condiciones financieras): {fmt(get_val('NFCI'), 3)} — {'RESTRICTIVO' if (not pd.isna(get_val('NFCI')) and get_val('NFCI') > 0) else 'LAXO'} — percentil {fmt(get_pct('NFCI'), 0)}/100
  • Spread High Yield (OAS): {fmt(get_val('HY_SPREAD'))} pb — percentil {fmt(get_pct('HY_SPREAD'), 0)}/100
  • Spread BAA-10Y: {fmt(get_val('SPREAD_BAA10Y'))} pp
  • M2 YoY: {fmt(get_val('M2_YOY'))}%

MERCADOS Y FX:
  • USD/MXN: {fmt(get_val('USDMXN'))} — percentil {fmt(get_pct('USDMXN'), 0)}/100
  • Índice Dólar (DXY): {fmt(get_val('DXY'))} — percentil {fmt(get_pct('DXY'), 0)}/100
  • Petróleo WTI: {fmt(get_val('WTI'))} USD/barril — percentil {fmt(get_pct('WTI'), 0)}/100
  • VIX: {fmt(get_val('VIX'))} — {'PÁNICO (>30)' if (not pd.isna(get_val('VIX')) and get_val('VIX') > 30) else ('ESTRÉS (20-30)' if (not pd.isna(get_val('VIX')) and get_val('VIX') > 20) else 'COMPLACENCIA (<20)')}

MÉXICO:
  • Inflación MX YoY: {fmt(get_val('MX_CPI_YOY'))}%
  • Tasa 3M México (proxy Banxico): {fmt(get_val('MX_3M'))}%
  • Bono 10Y México (M10): {fmt(get_val('MX_10Y'))}%
  • Spread soberano MX-US 10Y: {fmt(get_val('SPREAD_MX_US_10Y'))} pp
  • Curva MX 10Y-3M: {fmt(get_val('CURVE_MX_10Y3M'))} pp
"""

ranking_text = """

=================================================================
  SECCIÓN 2: PRONÓSTICO 5 DÍAS — RANKING FINAL
  (50% Monte Carlo + 30% Logístico + 20% kNN)
=================================================================

🏆 TOP 3 ACTIVOS CON MAYOR PROBABILIDAD DE SUBIR:
"""
for _, row in top3_up.iterrows():
    ranking_text += build_asset_paragraph(row['Ticker'])

ranking_text += """
🔻 BOTTOM 3 ACTIVOS CON MAYOR PROBABILIDAD DE BAJAR:
"""
for _, row in bottom3.iterrows():
    ranking_text += build_asset_paragraph(row['Ticker'])

risks_text = f"""

=================================================================
  SECCIÓN 3: RIESGOS PRINCIPALES DEL ESCENARIO
=================================================================

1. RIESGO DE TASAS: La Fed mantiene tasas {'elevadas' if (not pd.isna(get_val('EFFR')) and get_val('EFFR') > 4) else 'moderadas'} (EFFR={fmt(get_val('EFFR'))}%).
   Un dato de inflación sorpresivo (CPI/PCE) podría re-acelerar la curva y presionar activos
   de riesgo y bonos. La curva 10Y-2Y en {fmt(get_val('CURVE_10Y2Y'))} pb indica condiciones {'de inversión (recesión latente)' if (not pd.isna(get_val('CURVE_10Y2Y')) and get_val('CURVE_10Y2Y') < 0) else 'más normales'}.

2. RIESGO CREDITICIO: El spread High Yield en {fmt(get_val('HY_SPREAD'))} pb está en el percentil {fmt(get_pct('HY_SPREAD'), 0)} de los últimos 5 años.
   Un spread >500 pb históricamente correlaciona con deterioro en acciones de -3% a -8% en 5 días.

3. RIESGO CAMBIARIO (MXN): USD/MXN en {fmt(get_val('USDMXN'))} — percentil {fmt(get_pct('USDMXN'), 0)}/100.
   Un dólar fuerte (DXY alto) presiona materias primas y emergentes. Para activos en pesos,
   el riesgo de depreciación del MXN actúa como multiplicador negativo.

4. RIESGO DE VOLATILIDAD: VIX en {fmt(get_val('VIX'))}. {'VIX elevado implica mayor incertidumbre y potenciales drawdowns abruptos en activos de riesgo.' if (not pd.isna(get_val('VIX')) and get_val('VIX') > 20) else 'VIX contenido sugiere complacencia; shocks pueden ser abruptos si el mercado se sorprende.'}

5. RIESGO MACRO EUA: Desempleo en {fmt(get_val('UNRATE'))}%, Jobless Claims en {fmt(get_val('CLAIMS'), 0)}.
   Deterioro rápido del mercado laboral cambiaría el régimen de riesgo hacia activos defensivos.

6. RIESGO PETRÓLEO/ENERGÍA: WTI en {fmt(get_val('WTI'))} USD/barril.
   Shocks en el precio del crudo afectan inflación, costos corporativos y el MXN (dado que México
   es exportador de petróleo; WTI alto = MXN más fuerte, ceteris paribus).

=================================================================
  ADVERTENCIAS Y LIMITACIONES ESTADÍSTICAS
=================================================================

• Este sistema es PROBABILÍSTICO, no determinístico. Las probabilidades reflejan frecuencias
  históricas bajo condiciones similares, no garantías de resultados futuros.

• El modelo asume estacionariedad relativa del mercado. Eventos tail (shocks geopolíticos,
  crisis sistémicas, pandemias) pueden invalidar las distribuciones históricas.

• El VAR(1) asume relaciones lineales y estables entre variables endógenas. En regímenes
  de alta volatilidad, estas relaciones pueden romperse.

• Horizonte: 5 días hábiles. Más allá de este horizonte, la incertidumbre crece
  no linealmente y las probabilidades convergen a 50%.

• Los datos de FRED pueden tener hasta {LAG_BDAYS['M']} días hábiles de rezago (series mensuales).
  El "estado actual" puede no reflejar las últimas publicaciones macro.

SISTEMA GENERADO AUTOMÁTICAMENTE — {TODAY.strftime('%Y-%m-%d %H:%M')}
"""

CONCLUSION_TEXT = macro_text + ranking_text + risks_text

print('✅ Conclusiones generadas')
print(CONCLUSION_TEXT[:3000])

In [ ]:
# ============================================================
# CELDA 17: EXPORTAR EXCEL FINAL — MULTI-HOJA
# ============================================================
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def style_header(ws, row=1):
    """Aplica estilo oscuro a la fila de encabezado."""
    dark_fill = PatternFill('solid', fgColor='1F3864')
    white_font = Font(bold=True, color='FFFFFF', size=10)
    for cell in ws[row]:
        cell.fill = dark_fill
        cell.font = white_font
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

def auto_col_width(ws):
    """Ajusta ancho de columnas automáticamente."""
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
            except:
                pass
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 60)

def df_to_sheet(wb, sheet_name, df, index=True):
    """Escribe un DataFrame en una hoja del workbook con estilo."""
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
    # Reset index si se quiere incluir
    df_write = df.reset_index() if index else df.copy()
    # Escribir encabezado
    for c_idx, col_name in enumerate(df_write.columns, 1):
        ws.cell(1, c_idx, str(col_name))
    # Escribir datos
    for r_idx, row in enumerate(df_write.itertuples(index=False), 2):
        for c_idx, val in enumerate(row, 1):
            try:
                ws.cell(r_idx, c_idx, val if not (isinstance(val, float) and np.isnan(val)) else None)
            except:
                ws.cell(r_idx, c_idx, str(val))
    style_header(ws)
    auto_col_width(ws)
    return ws

# Crear workbook
wb = openpyxl.Workbook()
wb.remove(wb.active)  # Eliminar hoja por defecto

print('📝 Escribiendo hojas Excel...')

# 1. PRICES_YAHOO_DAILY
prices_out = prices_df[prices_df.index >= FOCUS_START].copy()
prices_out.columns = [FULL_NAME.get(c, c) for c in prices_out.columns]
df_to_sheet(wb, 'PRICES_YAHOO_DAILY', prices_out)
print('  ✅ PRICES_YAHOO_DAILY')

# 2. MACRO_FRED_RAW
macro_out = macro_df[macro_df.index >= FOCUS_START].copy()
df_to_sheet(wb, 'MACRO_FRED_RAW', macro_out)
print('  ✅ MACRO_FRED_RAW')

# 3. FEATURES_DAILY
df_to_sheet(wb, 'FEATURES_DAILY', df_feat)
print('  ✅ FEATURES_DAILY')

# 4. FORWARD_5D
fwd5_out = df_fwd5.copy()
fwd5_out.columns = [FULL_NAME.get(c, c) for c in fwd5_out.columns]
df_to_sheet(wb, 'FORWARD_5D', fwd5_out)
print('  ✅ FORWARD_5D')

# 5. VOLATILIDAD_SECTORES
df_to_sheet(wb, 'VOLATILIDAD_SECTORES', vol_ranking.reset_index(), index=False)
print('  ✅ VOLATILIDAD_SECTORES')

# 6. CORR_FEATURE_TO_5D
corr_out = corr_matrix.copy()
corr_out.columns = [FULL_NAME.get(c, c) for c in corr_out.columns]
df_to_sheet(wb, 'CORR_FEATURE_TO_5D', corr_out)
print('  ✅ CORR_FEATURE_TO_5D')

# 7. TOP_CORR_POS
df_to_sheet(wb, 'TOP_CORR_POS', top_pos, index=False)
print('  ✅ TOP_CORR_POS')

# 8. TOP_CORR_NEG
df_to_sheet(wb, 'TOP_CORR_NEG', top_neg, index=False)
print('  ✅ TOP_CORR_NEG')

# 9. MINMAX_CORR_BY_IND
df_to_sheet(wb, 'MINMAX_CORR_BY_IND', minmax_by_ind, index=False)
print('  ✅ MINMAX_CORR_BY_IND')

# 10. DEFINICIONES
df_to_sheet(wb, 'DEFINICIONES', defs_df, index=False)
print('  ✅ DEFINICIONES')

# 11. KNN_SIMILAR_DAYS
df_to_sheet(wb, 'KNN_SIMILAR_DAYS', knn_df, index=False)
print('  ✅ KNN_SIMILAR_DAYS')

# 12. LOGISTIC_PROBS
logit_drivers_all = []
for asset, drv_df in logit_drivers.items():
    d = drv_df.copy()
    d.insert(0, 'Activo', FULL_NAME.get(asset, asset))
    logit_drivers_all.append(d)
logit_full = pd.concat([logit_df] + (logit_drivers_all if logit_drivers_all else [pd.DataFrame()]), axis=0, ignore_index=True)
df_to_sheet(wb, 'LOGISTIC_PROBS', logit_df, index=False)
print('  ✅ LOGISTIC_PROBS')

# 12b. LOGISTIC_DRIVERS
if logit_drivers_all:
    drivers_concat = pd.concat(logit_drivers_all, ignore_index=True)
    df_to_sheet(wb, 'LOGISTIC_DRIVERS', drivers_concat, index=False)
    print('  ✅ LOGISTIC_DRIVERS')

# 13. MONTE_CARLO
df_to_sheet(wb, 'MONTE_CARLO', mc_df, index=False)
print('  ✅ MONTE_CARLO')

# 14. RANKING_FINAL
df_to_sheet(wb, 'RANKING_FINAL', ranking_df, index=False)
print('  ✅ RANKING_FINAL')

# 15. READOUT_HOY
df_to_sheet(wb, 'READOUT_HOY', readout_df, index=False)
print('  ✅ READOUT_HOY')

# 16. CONCLUSION (texto largo)
ws_conc = wb.create_sheet('CONCLUSION')
lines = CONCLUSION_TEXT.split('\n')
for r_idx, line in enumerate(lines, 1):
    ws_conc.cell(r_idx, 1, line)
ws_conc.column_dimensions['A'].width = 120
print('  ✅ CONCLUSION')

# 17. MISSING_REPORT
missing_rows = []
for k in missing_yahoo:
    missing_rows.append({'Fuente':'Yahoo Finance','Clave':k,'Nombre':FULL_NAME.get(k,''),'Razon':'No se pudo descargar (todos los tickers fallaron)'})
for k in missing_fred:
    missing_rows.append({'Fuente':'FRED','Clave':k,'Nombre':k,'Razon':'Ningún ID de FRED respondió correctamente'})
if var_fallback_msg:
    missing_rows.append({'Fuente':'Modelo','Clave':'VAR1','Nombre':'VAR(1)','Razon':var_fallback_msg})
missing_report_df = pd.DataFrame(missing_rows) if missing_rows else pd.DataFrame(columns=['Fuente','Clave','Nombre','Razon'])
df_to_sheet(wb, 'MISSING_REPORT', missing_report_df, index=False)
print('  ✅ MISSING_REPORT')

# Guardar
wb.save(EXCEL_FILE)
print(f'\n✅✅✅ Excel guardado: {EXCEL_FILE}')
print(f'   Hojas: {wb.sheetnames}')

# Descargar en Colab
try:
    from google.colab import files
    files.download(EXCEL_FILE)
    print('⬇️  Descarga iniciada automáticamente')
except ImportError:
    print(f'💡 Archivo guardado en el directorio actual: {EXCEL_FILE}')
    print('   Para descargar en Colab: from google.colab import files; files.download(EXCEL_FILE)')

In [ ]:
# ============================================================
# CELDA 18: QA — VALIDACIONES FINALES
# ============================================================

print('='*60)
print('🔎 VALIDACIONES FINALES QA')
print('='*60)

errors = []
warnings_qa = []

# 1. Suficientes datos por activo en 5 años
print('\n1. Cobertura de datos por activo (últimos 5 años):')
for k in ASSET_KEYS:
    if k not in prices_df.columns:
        errors.append(f'{k}: NO descargado')
        continue
    n = prices_df[k][prices_df.index >= FOCUS_START].notna().sum()
    status = '✅' if n > 800 else ('⚠️' if n > 400 else '❌')
    print(f'  {status} {FULL_NAME.get(k, k):45s}: {n} días')
    if n < 400:
        errors.append(f'{k}: solo {n} días en ventana 5y')

# 2. Forward returns no son todos NaN
print('\n2. Forward returns 5d (NaN check):')
for k in ASSET_KEYS:
    if k in df_fwd5.columns:
        n_valid = df_fwd5[k].notna().sum()
        status = '✅' if n_valid > 200 else '⚠️'
        print(f'  {status} {k}: {n_valid} valores válidos')
        if n_valid < 50:
            warnings_qa.append(f'{k}: pocos forward returns ({n_valid})')

# 3. Alineación de fechas
print(f'\n3. Alineación de fechas:')
print(f'  Features index : {df_feat.index[0].date()} → {df_feat.index[-1].date()} ({len(df_feat)} días)')
print(f'  Fwd5 index     : {df_fwd5.index[0].date()} → {df_fwd5.index[-1].date()} ({len(df_fwd5)} días)')
if abs(len(df_feat) - len(df_fwd5)) > 10:
    warnings_qa.append('Diferencia notable en longitud de feat vs fwd5')
else:
    print('  ✅ Indexing correcto')

# 4. Monte Carlo
print(f'\n4. Monte Carlo:')
print(f'  Total simulaciones: {total_sim:,} ({"✅" if total_sim >= 1_000_000 else "⚠️ MENOS DE 1M"})')
print(f'  Retornos almacenados: {stored:,} filas x {N_ASSETS_MC} activos')

# 5. Resumen de resultados
print(f'\n5. Resumen ranking final:')
print(ranking_df[['Rank','Nombre','P_Combinada_%','Convergencia']].head(10).to_string())

print(f'\n{'='*60}')
if errors:
    print(f'❌ ERRORES ({len(errors)}):')
    for e in errors: print(f'   - {e}')
else:
    print('✅ SIN ERRORES CRÍTICOS')

if warnings_qa:
    print(f'⚠️  ADVERTENCIAS ({len(warnings_qa)}):')
    for w in warnings_qa: print(f'   - {w}')
else:
    print('✅ SIN ADVERTENCIAS')

print(f'\n🎉 SISTEMA COMPLETADO EXITOSAMENTE')
print(f'   Excel: {EXCEL_FILE}')
print(f'   {len(wb.sheetnames)} hojas generadas')
print(f'   {total_sim:,} simulaciones Monte Carlo')
print(f'   {TODAY.strftime("%Y-%m-%d")}')